# Notebook 3 - DistilBERT Classifier Training

Now that I have a labelled dataset I can train the classifier.
The model is `distilbert-base-uncased` - a smaller, faster version of BERT (~66M parameters).
I fine-tune it to output a probability: how likely is this prompt a jailbreak attempt?

## Step 1 - Imports

- **PyTorch** - training loop, tensors, GPU support
- **HuggingFace Transformers** - pre-trained DistilBERT weights and tokeniser
- **scikit-learn** - F1, AUC-ROC metrics after training

In [1]:
import json
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
import matplotlib
matplotlib.use('Agg')  # save without display window
import matplotlib.pyplot as plt

## Step 2 - Config and paths

All training settings in one place so they're easy to change and easy to see at a glance.
- `EPOCHS = 3` - enough for DistilBERT to converge without overfitting on 1200 examples
- `BATCH = 32` - fits comfortably in GPU memory
- `LR = 2e-5` - standard fine-tuning rate for BERT-family models

In [2]:
ROOT_DIR       = Path.cwd().parent
DATA_PROCESSED = ROOT_DIR / 'data' / 'processed'
CLASSIFIER_DIR = ROOT_DIR / 'models' / 'classifier'
FIGURES_DIR    = ROOT_DIR / 'data' / 'figures'

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS  = 3
BATCH   = 32
MAX_LEN = 128
LR      = 2e-5

torch.manual_seed(42)
print(f'Using device: {DEVICE}')

Using device: cpu


## Step 3 - Load the splits

These were saved by notebook 2. I just read them back here - no re-sampling, no re-shuffling, so the train/val/test split is identical to what was saved.

In [3]:
def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_data = load_jsonl(DATA_PROCESSED / 'classifier_train.jsonl')
val_data   = load_jsonl(DATA_PROCESSED / 'classifier_val.jsonl')
test_data  = load_jsonl(DATA_PROCESSED / 'classifier_test.jsonl')

print(f'Train: {len(train_data)}')
print(f'Val:   {len(val_data)}')
print(f'Test:  {len(test_data)}')

Train: 1200
Val:   150
Test:  200


## Step 4 - PyTorch Dataset class

PyTorch's `DataLoader` needs a `Dataset` object - something it can index into with `[i]`.
This class wraps my list of dicts and tokenises each prompt on the fly when the DataLoader asks for it.

Why tokenise here and not upfront? Memory. Storing 1200 pre-tokenised tensors uses more RAM than tokenising one batch at a time.

In [4]:
class PromptDataset(Dataset):
    def __init__(self, data, tokeniser):
        self.data      = data
        self.tokeniser = tokeniser

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i]
        enc  = self.tokeniser(
            item['text'],
            truncation=True,
            max_length=MAX_LEN,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(item['label'], dtype=torch.long),
        }

## Step 5 - Load the tokeniser and create DataLoaders

The tokeniser converts raw text to token IDs that DistilBERT understands.
I use the same checkpoint name (`distilbert-base-uncased`) so the vocab matches the model weights.

`shuffle=True` on the train loader so the model doesn't see examples in the same order each epoch.

In [5]:
tok = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_loader = DataLoader(PromptDataset(train_data, tok), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(PromptDataset(val_data,   tok), batch_size=BATCH)
test_loader  = DataLoader(PromptDataset(test_data,  tok), batch_size=BATCH)

print('DataLoaders ready')

DataLoaders ready


## Step 6 - Load the pre-trained model

I load the DistilBERT weights from HuggingFace and add a 2-class classification head on top.
The pre-trained weights already understand English - fine-tuning just teaches it to distinguish
jailbreak prompts from benign ones.

The scheduler gradually reduces the learning rate over training - helps the model settle without overshooting.

In [6]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(0.1 * total_steps)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded - {n_params:.0f}M parameters')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded - 67M parameters


## Step 7 - Evaluation helper

I need to check the model on the val set after each epoch to see if it's improving.
This function runs all batches through the model with `torch.no_grad()` (no gradient tracking needed - it's not training) and returns the true labels, predictions, and raw probabilities.

The `tau` parameter is the decision threshold - above `tau` - predicted jailbreak.

In [7]:
def evaluate(loader, tau=0.5):
    model.eval()
    all_labels = []
    all_probs  = []

    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            out  = model(ids, mask)
            probs = torch.softmax(out.logits, dim=-1)[:, 1]  # probability of class 1 (jailbreak)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch['labels'].numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds      = (all_probs >= tau).astype(int)
    return all_labels, preds, all_probs

## Step 8 - Training loop

Standard fine-tuning loop:
1. Forward pass -> compute loss
2. Backward pass -> compute gradients
3. Clip gradients (stops them exploding if one batch is unusual)
4. Update weights -> step the optimizer and scheduler

After each epoch I run `evaluate()` on the val set to track progress.

In [8]:
train_losses = []
val_f1s      = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for batch in train_loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        out = model(ids, mask, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += out.loss.item()

    avg_loss          = total_loss / len(train_loader)
    y, preds, probs   = evaluate(val_loader)
    val_f1  = f1_score(y, preds)
    val_auc = roc_auc_score(y, probs)
    print(f'Epoch {epoch}  loss={avg_loss:.4f}  val_F1={val_f1:.4f}  val_AUC={val_auc:.4f}')
    train_losses.append(avg_loss)
    val_f1s.append(val_f1)

Epoch 1  loss=0.4147  val_F1=0.9870  val_AUC=0.9943
Epoch 2  loss=0.0771  val_F1=0.9870  val_AUC=0.9998
Epoch 3  loss=0.0479  val_F1=0.9870  val_AUC=0.9998


## Step 9 - Test set results

The val set was used to monitor training - it's been indirectly influencing decisions (like how many epochs to run). The **test set** has never been seen before, so it gives an honest view of how the model will perform on new prompts.

In [9]:
y, preds, probs = evaluate(test_loader)

print('=== Test set results (tau = 0.5) ===')
print(classification_report(y, preds, target_names=['benign', 'jailbreak']))
print(f'AUC-ROC: {roc_auc_score(y, probs):.4f}')

=== Test set results (tau = 0.5) ===
              precision    recall  f1-score   support

      benign       1.00      0.99      1.00       101
   jailbreak       0.99      1.00      0.99        99

    accuracy                           0.99       200
   macro avg       0.99      1.00      0.99       200
weighted avg       1.00      0.99      1.00       200

AUC-ROC: 1.0000


## Step 9b - Save training curves, ROC curve and confusion matrix

Saving these figures to `reports/figures/` so they can be used directly in the dissertation without re-running the full notebook.

In [10]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- Training curves ---
epochs_x = list(range(1, EPOCHS + 1))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(epochs_x, train_losses, marker='o', color='steelblue')
axes[0].set_title('Training Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_xticks(epochs_x)

axes[1].plot(epochs_x, val_f1s, marker='o', color='darkorange')
axes[1].set_title('Validation F1 per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_xticks(epochs_x)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved training_curves.png')

# --- ROC curve ---
fpr_vals, tpr_vals, _ = roc_curve(y, probs)
auc_score = roc_auc_score(y, probs)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_vals, tpr_vals, color='steelblue', lw=2,
        label=f'ROC curve (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — DistilBERT Classifier')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved roc_curve.png')

# --- Confusion matrix ---
cm = confusion_matrix(y, preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Benign', 'Jailbreak'])
ax.set_yticklabels(['Benign', 'Jailbreak'])
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix — Test Set (tau=0.5)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved confusion_matrix.png')


Saved training_curves.png
Saved roc_curve.png
Saved confusion_matrix.png


## Step 10 - Threshold sensitivity

The default threshold `tau = 0.5` means: block if jailbreak probability â‰¥ 50%.
But that's just a starting point. Here I try 5 values to see the trade-off:

- **Lower tau** -> blocks more - catches more jailbreaks but also blocks more safe prompts (higher FPR)
- **Higher tau** -> blocks less - fewer false positives but lets more jailbreaks through (higher FNR)

I picked `tau = 0.5` because F1 is good and FPR is acceptable.

In [11]:
y_v, _, p_v = evaluate(val_loader)

print(f'{"tau":>5}  {"F1":>7}  {"FNR":>7}  {"FPR":>7}')
for tau in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = (p_v >= tau).astype(int)
    tp  = int(np.sum((y_v == 1) & (preds == 1)))
    fn  = int(np.sum((y_v == 1) & (preds == 0)))
    fp  = int(np.sum((y_v == 0) & (preds == 1)))
    tn  = int(np.sum((y_v == 0) & (preds == 0)))
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    print(f'{tau:>5.1f}  {f1_score(y_v, preds):>7.4f}  {fnr:>7.4f}  {fpr:>7.4f}')

  tau       F1      FNR      FPR
  0.3   0.9870   0.0000   0.0270
  0.4   0.9870   0.0000   0.0270
  0.5   0.9870   0.0000   0.0270
  0.6   0.9870   0.0000   0.0270
  0.7   0.9870   0.0000   0.0270


## Step 11 - Save the model

Notebook 4 needs to load this classifier without re-training it.
`save_pretrained` saves the weights and the tokeniser config together - so I can reload everything with a single `from_pretrained` call.

In [12]:
# CLASSIFIER_DIR.mkdir(parents=True, exist_ok=True)
# model.save_pretrained(str(CLASSIFIER_DIR), safe_serialization=False)
# tok.save_pretrained(str(CLASSIFIER_DIR))
# print(f'Model saved -> {CLASSIFIER_DIR}')


import gc

CLASSIFIER_DIR.mkdir(parents=True, exist_ok=True)

# Delete old safetensors first — Windows won't overwrite a memory-mapped file (OS error 1224)
_old = CLASSIFIER_DIR / 'model.safetensors'
if _old.exists():
    try:
        _old.unlink()
    except PermissionError:
        print('ERROR: model.safetensors is locked by another process.')
        print('Close any other kernel or app that loaded the model, then re-run this cell.')
        raise

gc.collect()
model.save_pretrained(str(CLASSIFIER_DIR), safe_serialization=False)
tok.save_pretrained(str(CLASSIFIER_DIR))
print(f'Model saved -> {CLASSIFIER_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved -> d:\National College Of Ireland\SEM-2\Dissertation\AdaptivePromptGuard\models\classifier
